In [1]:
!pip install -q pyspark


In [2]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [3]:
# Create Data Lake directory structure
base_path = "/content/data_lake"
bronze_path = os.path.join(base_path, "bronze")
silver_path = os.path.join(base_path, "silver")
silver_output_path = os.path.join(silver_path, "preprocessed_sales")

os.makedirs(bronze_path, exist_ok=True)
os.makedirs(silver_path, exist_ok=True)

print("Data Lake structure created:")
print(base_path)
print(bronze_path)
print(silver_path)

Data Lake structure created:
/content/data_lake
/content/data_lake/bronze
/content/data_lake/silver


In [4]:
source_file = "/content/sales.csv.xls"
bronze_file = os.path.join(bronze_path, "sales.csv")

shutil.copy2(source_file, bronze_file)

print("Bronze file created:")
print(bronze_file)

Bronze file created:
/content/data_lake/bronze/sales.csv


In [5]:
print("Bronze file exists:", os.path.exists(bronze_file))
print("Bronze file size:", os.path.getsize(bronze_file), "bytes")

Bronze file exists: True
Bronze file size: 109537 bytes


In [6]:
spark = (
    SparkSession.builder
    .appName("SalesDataLakePreprocessing")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 4.0.4


In [7]:
sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(bronze_file)
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [8]:
sales_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [9]:
sales_df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [10]:
initial_count = sales_df.count()

print("Total records before preprocessing:", initial_count)

Total records before preprocessing: 1000


In [11]:
duplicate_count = initial_count - sales_df.dropDuplicates().count()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 10


In [12]:
sales_df.groupBy(sales_df.columns).count() \
    .filter(F.col("count") > 1) \
    .show(20, truncate=False)

+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits Pack  |Grocery        |7      

In [13]:
null_counts = sales_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in sales_df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [14]:
text_columns = [
    field.name
    for field in sales_df.schema.fields
    if isinstance(field.dataType, StringType)
]

print("Text columns:", text_columns)

Text columns: ['customer_id', 'customer_name', 'product', 'category', 'payment_method', 'city', 'state', 'order_date', 'order_status']


In [15]:
for column in text_columns:
    print(f"\nDistinct values in {column}:")
    sales_df.select(column).distinct().show(30, truncate=False)


Distinct values in customer_id:
+-----------+
|customer_id|
+-----------+
|C0114      |
|C0141      |
|C0243      |
|C0034      |
|C0154      |
|C0347      |
|C0130      |
|C0338      |
|C0167      |
|C0179      |
|C0195      |
|C0150      |
|C0033      |
|C0107      |
|C0016      |
|C0131      |
|C0191      |
|C0261      |
|C0057      |
|C0280      |
|C0046      |
|C0284      |
|C0102      |
|C0061      |
|C0337      |
|C0028      |
|C0058      |
|C0050      |
|C0139      |
|C0109      |
+-----------+
only showing top 30 rows

Distinct values in customer_name:
+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pill

In [16]:
numeric_check_columns = [
    c for c in ["quantity", "unit_price", "discount_percentage", "discount_percent", "discount"]
    if c in sales_df.columns
]

print("Numerical fields found:", numeric_check_columns)

Numerical fields found: ['quantity', 'unit_price', 'discount_percent']


In [17]:
sales_df.select(numeric_check_columns).describe().show()

+-------+------------------+------------------+------------------+
|summary|          quantity|        unit_price|  discount_percent|
+-------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|
|   mean|             5.355|            5702.4|            19.948|
| stddev|2.9916425128890194|10990.194484849431|14.236782711763993|
|    min|                -2|              -500|                -5|
|    max|                10|             81010|               120|
+-------+------------------+------------------+------------------+



In [18]:
for column in numeric_check_columns:
    print(f"\nPotentially unusual values in {column}:")
    sales_df.select(column).orderBy(F.col(column)).show(20, truncate=False)


Potentially unusual values in quantity:
+--------+
|quantity|
+--------+
|-2      |
|-2      |
|-2      |
|-2      |
|-2      |
|-1      |
|0       |
|0       |
|0       |
|0       |
|0       |
|0       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
|1       |
+--------+
only showing top 20 rows

Potentially unusual values in unit_price:
+----------+
|unit_price|
+----------+
|-500      |
|-500      |
|-500      |
|-250      |
|0         |
|0         |
|0         |
|0         |
|190       |
|210       |
|210       |
|230       |
|240       |
|250       |
|250       |
|250       |
|250       |
|260       |
|260       |
|270       |
+----------+
only showing top 20 rows

Potentially unusual values in discount_percent:
+----------------+
|discount_percent|
+----------------+
|-5              |
|-5              |
|-5              |
|-5              |
|0               |
|0               |
|0               |
|0               |
|0               |
|0           

In [19]:
date_columns = [
    c for c in sales_df.columns
    if "date" in c.lower()
]

print("Date-related columns:", date_columns)

Date-related columns: ['order_date']


In [20]:
for column in date_columns:
    print(f"\nSample values from {column}:")
    sales_df.select(column).distinct().show(30, truncate=False)


Sample values from order_date:
+----------+
|order_date|
+----------+
|2026-02-01|
|2025-06-03|
|2025-02-25|
|2026-02-27|
|2025-08-17|
|2026-07-23|
|2026-07-04|
|2025-09-14|
|2025-03-20|
|2025-04-05|
|2026-04-15|
|2026-06-05|
|2025-01-03|
|2025-12-07|
|2026-03-15|
|2025-12-30|
|2025-08-18|
|2026-09-11|
|2025-06-16|
|2026-09-06|
|2025-07-15|
|2026-08-24|
|2025/07/14|
|2025-05-06|
|2025-04-12|
|2026-02-15|
|2025-07-21|
|2025-05-23|
|2025-10-09|
|2026-01-05|
+----------+
only showing top 30 rows


In [21]:
# Start Silver processing again from the ORIGINAL Bronze DataFrame.
# This removes the old date-parsing transformation from the Spark lineage.

silver_df = sales_df.dropDuplicates()

print("Silver preprocessing started from the original Bronze DataFrame.")
print("Rows after removing exact duplicates:", silver_df.count())

Silver preprocessing started from the original Bronze DataFrame.
Rows after removing exact duplicates: 990


In [22]:
text_columns = [
    "customer_id",
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for column in text_columns:
    silver_df = silver_df.withColumn(
        column,
        F.trim(F.col(column))
    )

print("Extra spaces removed from text fields.")

Extra spaces removed from text fields.


In [23]:
categorical_columns = [
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for column in categorical_columns:
    silver_df = silver_df.withColumn(
        column,
        F.initcap(F.lower(F.trim(F.col(column))))
    )

print("Categorical text capitalization standardized.")

Categorical text capitalization standardized.


In [24]:
numeric_columns = [
    "quantity",
    "unit_price",
    "discount_percent"
]

for column in numeric_columns:
    median_value = silver_df.approxQuantile(
        column,
        [0.5],
        0.01
    )[0]

    silver_df = silver_df.fillna({
        column: median_value
    })

# Handle missing text values
silver_df = silver_df.fillna(
    "Unknown",
    subset=text_columns
)

print("Missing values handled.")

Missing values handled.


In [25]:
silver_df = silver_df.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity"))
     .otherwise(None)
)

In [26]:
silver_df = silver_df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") >= 0, F.col("unit_price"))
     .otherwise(None)
)

In [27]:
silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent") >= 0) &
        (F.col("discount_percent") <= 100),
        F.col("discount_percent")
    ).otherwise(None)
)

In [28]:
for column in numeric_columns:
    median_value = silver_df.approxQuantile(
        column,
        [0.5],
        0.01
    )[0]

    silver_df = silver_df.fillna({
        column: median_value
    })

print("Numerical validation completed.")

Numerical validation completed.


In [29]:
# Safely parse order_date.
# Invalid dates such as 31/02/2026 become NULL.

date_column = "order_date"

raw_date = F.trim(
    F.col(date_column).cast("string")
)

parsed_date = F.coalesce(
    F.to_date(
        F.try_to_timestamp(raw_date, F.lit("yyyy-MM-dd"))
    ),
    F.to_date(
        F.try_to_timestamp(raw_date, F.lit("dd-MM-yyyy"))
    ),
    F.to_date(
        F.try_to_timestamp(raw_date, F.lit("dd/MM/yyyy"))
    ),
    F.to_date(
        F.try_to_timestamp(raw_date, F.lit("MM/dd/yyyy"))
    ),
    F.to_date(
        F.try_to_timestamp(raw_date, F.lit("yyyy/MM/dd"))
    ),
    F.to_date(
        F.try_to_timestamp(raw_date, F.lit("MM-dd-yyyy"))
    )
)

silver_df = silver_df.withColumn(
    date_column,
    parsed_date
)

print("Date validation completed.")
print("Invalid dates have been converted to NULL.")

Date validation completed.
Invalid dates have been converted to NULL.


In [30]:
silver_df.filter(
    F.col("order_date").isNull()
).select(
    "order_id",
    "order_date"
).show(20, truncate=False)

+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |NULL      |
|100637  |NULL      |
|100448  |NULL      |
|100634  |NULL      |
+--------+----------+



In [31]:
for column in numeric_columns:
    median_value = silver_df.approxQuantile(
        column,
        [0.5],
        0.01
    )[0]

    silver_df = silver_df.fillna({
        column: median_value
    })

print("Final missing numerical values handled.")

Final missing numerical values handled.


In [32]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = false)



In [33]:
silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [34]:
final_count = silver_df.count()

print("Rows before preprocessing:", initial_count)
print("Rows after preprocessing:", final_count)
print("Rows removed:", initial_count - final_count)

Rows before preprocessing: 1000
Rows after preprocessing: 990
Rows removed: 10


In [35]:
silver_spark_path = "/content/data_lake/silver/preprocessed_sales"

(
    silver_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(silver_spark_path)
)

print("Silver data saved successfully.")

Silver data saved successfully.


In [36]:
import glob
import shutil
import os

# Find the CSV file generated by Spark
part_files = glob.glob(
    os.path.join(silver_spark_path, "part-*.csv")
)

print("Spark output files found:", len(part_files))

# Create the required single Silver CSV file
silver_csv = "/content/sales_silver.csv"
shutil.copy2(part_files[0], silver_csv)

print("Silver CSV created successfully:")
print(silver_csv)

Spark output files found: 1
Silver CSV created successfully:
/content/sales_silver.csv


In [37]:
verified_silver_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(silver_csv)
)

print("Silver row count:", verified_silver_df.count())

verified_silver_df.show(10, truncate=False)

Silver row count: 990
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive 

In [38]:
print("Original Bronze row count:", initial_count)
print("Final Silver row count:", final_count)
print("Verified Silver row count:", verified_silver_df.count())

Original Bronze row count: 1000
Final Silver row count: 990
Verified Silver row count: 990


## Final Verification and Summary

The Bronze layer contains the original `sales.csv` dataset without modification.

The Silver layer was created using Apache Spark. The preprocessing included:

- Removing exact duplicate rows.
- Trimming extra spaces from text fields.
- Standardizing capitalization of categorical text fields.
- Handling missing numeric values using median imputation.
- Replacing missing text values with `Unknown`.
- Validating quantity to keep positive values.
- Validating unit price to keep non-negative values.
- Validating discount percentage to keep values between 0 and 100.
- Converting recognized date formats into a consistent date format.
- Converting invalid dates to NULL while preserving the corresponding sales records.
- Saving the processed dataset as CSV in the Silver layer.

The row counts before and after preprocessing were calculated dynamically using Spark.
The Silver output was read back using Spark to verify the saved data.